# Quickstart

Get a guild running in 5 lines.

**Prerequisites:** `pip install guildmaster-ai[openrouter]` and set `OPENROUTER_API_KEY` in your environment or `.env` file.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads OPENROUTER_API_KEY from .env

In [ ]:
from guildmaster_ai import GeneralAdventurer, GuildBuilder

guild = (
    GuildBuilder().with_llm_provider("openrouter").register_adventurer(GeneralAdventurer).build()
)

result = await guild.run_quest("What are the three primary colors?")
print(result.summary)

## What just happened?

1. `GuildBuilder` created a guild with an OpenRouter LLM
2. A `GeneralAdventurer` was registered (it can handle any quest)
3. `run_quest()` ran the full lifecycle: **Plan** -> **Execute** -> **Verify** -> **Archive**
4. The result is a `QuestResult` with `success`, `summary`, and metadata

## Background quests

`run_quest()` is the submit-and-wait convenience. Under the hood every quest
runs in the background: `post_quest()` returns a `QuestTicket` immediately,
the receptionist reports status by UUID, and `wait_for_quest()` awaits the result.


In [ ]:
# Submit without blocking — execution happens in the background
ticket = await guild.post_quest("Name one famous lighthouse.")
print(f"Submitted: {ticket.quest_id} ({ticket.status})")

# Ask the receptionist for status at any time
report = guild.receptionist.check_status(ticket.quest_id)
print(f"Status: {report.status} (finished={report.finished})")

# Await the final result when you need it
result = await guild.wait_for_quest(ticket.quest_id)
print(result.summary)


In [ ]:
# Inspect the guild state
print(repr(guild))
print(f"Quests completed: {guild.info.archived}")
print(f"Adventurers: {[p.name or p.id for p in guild.roster]}")

## Using a different provider

Pass any LangChain `BaseChatModel` directly:

In [ ]:
# Example: pass a pre-configured model
from guildmaster_ai.llm.openrouter import ChatOpenRouter

custom_llm = ChatOpenRouter(model="meta-llama/llama-4-scout", temperature=0.3)

guild2 = GuildBuilder().with_llm_provider(custom_llm).register_adventurer(GeneralAdventurer).build()

result = await guild2.run_quest("Capital of France? One word.")
print(result.summary)